# Mat & Comp – RN 7: Transferencia de aprendizaje
## Aprovechar una red que ya aprendió a ver

**Curso:** Redes neuronales (Sección 4) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** RN 6

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar la idea de **transferencia de aprendizaje**: reutilizar lo que una red aprendió en una tarea para resolver otra con pocos datos.
2. Cargar una **red preentrenada** de `torchvision` (ResNet-18) y usarla para clasificar fotos.
3. Usar la red como **extractor de características**: convertir cada foto en 512 números y entrenar solo una capa nueva encima.
4. **Congelar** capas en PyTorch (`requires_grad = False`) y reemplazar la última capa.
5. Comparar con números una red entrenada desde cero y una red con transferencia cuando hay **pocas fotos**.

## 🔍 ¿Cómo aprendes a reconocer un animal nuevo?

Si nunca has visto un **okapi** y te muestran tres fotos, probablemente lo reconocerás en la cuarta. No necesitas 6000 fotos, como la red de RN 4 necesitó para cada dígito. ¿Por qué?

✏️ *¿Qué cosas ya sabías hacer antes de ver el okapi, que te ayudaron a aprenderlo tan rápido? Piensa en bordes, texturas, patas, ojos…*

**Mi respuesta:**

Tú ya sabes **ver**: reconoces bordes, texturas, formas, partes de animales. Solo te falta aprender qué combinación corresponde al okapi. Las redes pueden hacer lo mismo: tomamos una red que ya aprendió a ver con más de un millón de fotos y le enseñamos solo **la última parte**. Eso es la **transferencia de aprendizaje**.

## 1. La idea con dígitos

Primero, un experimento pequeño que hacemos de principio a fin. Entrenamos los «ojos» de una red convolucional (las capas de RN 5) con los dígitos **0 a 4**. Luego queremos reconocer los dígitos **5 a 9**, que la red nunca vio, con solo **10 ejemplos** de cada uno.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import LogisticRegression
import time

mnist = datasets.MNIST("datos", train=True, download=True, transform=transforms.ToTensor())
mnist_prueba = datasets.MNIST("datos", train=False, download=True, transform=transforms.ToTensor())
y_mnist = mnist.targets.numpy()
y_mnist_prueba = mnist_prueba.targets.numpy()
criterio = nn.CrossEntropyLoss()

Los «ojos» son los dos bloques convolucionales de RN 5. La «cabeza» es la capa `Linear` que decide la respuesta. Entrenamos ojos y cabeza juntos con los dígitos 0–4 (unas 30 000 imágenes, una época con Adam, como en RN 6):

In [ ]:
def crear_ojos():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),                      # sale una fila de 32 · 7 · 7 = 1568 números
    )

torch.manual_seed(0)
ojos = crear_ojos()
red_04 = nn.Sequential(ojos, nn.Linear(1568, 5))     # cabeza para 5 dígitos

solo_04 = Subset(mnist, np.where(y_mnist < 5)[0])
optimizador = optim.Adam(red_04.parameters(), lr=0.001)
red_04.train()
for X, y in DataLoader(solo_04, batch_size=64, shuffle=True):
    perdida = criterio(red_04(X), y)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

X, y = next(iter(DataLoader(Subset(mnist_prueba, np.where(y_mnist_prueba < 5)[0]), batch_size=10000)))
red_04.eval()
with torch.no_grad():
    print(f"Exactitud con los dígitos 0–4: {(red_04(X).argmax(dim=1) == y).float().mean():.1%}")

Ahora la tarea nueva: los dígitos 5–9, con solo 10 fotos de cada uno (50 en total). Restamos 5 a las etiquetas para que vayan de 0 a 4.

In [ ]:
rng = np.random.default_rng(0)
idx_pocos = np.concatenate([rng.choice(np.where(y_mnist == d)[0], 10, replace=False) for d in range(5, 10)])
X_pocos, y_pocos = next(iter(DataLoader(Subset(mnist, idx_pocos), batch_size=50)))
y_pocos = y_pocos - 5

idx_nuevos_prueba = np.where(y_mnist_prueba >= 5)[0]
X_np, y_np = next(iter(DataLoader(Subset(mnist_prueba, idx_nuevos_prueba), batch_size=10000)))
y_np = y_np - 5
print("Fotos para aprender:", X_pocos.shape[0], "  fotos de prueba:", X_np.shape[0])

Comparamos tres formas de aprender con esas 50 fotos:

- **A. Regresión logística sobre los píxeles** (ML 8): cada foto son 784 números.
- **B. Red convolucional desde cero**, entrenada solo con las 50 fotos.
- **C. Transferencia:** pasamos cada foto por los **ojos ya entrenados** (sin cambiarlos) y usamos sus 1568 números como entrada de una regresión logística.

In [ ]:
# A. Píxeles + regresión logística
A = LogisticRegression(max_iter=2000).fit(X_pocos.reshape(50, -1).numpy(), y_pocos.numpy())
exa_A = A.score(X_np.reshape(len(X_np), -1).numpy(), y_np.numpy())

# B. Red desde cero (100 pasos de Adam con las 50 fotos)
torch.manual_seed(0)
red_B = nn.Sequential(crear_ojos(), nn.Linear(1568, 5))
optimizador = optim.Adam(red_B.parameters(), lr=0.001)
for paso in range(100):
    perdida = criterio(red_B(X_pocos), y_pocos)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
with torch.no_grad():
    exa_B = (red_B(X_np).argmax(dim=1) == y_np).float().mean().item()

# C. Ojos entrenados con 0–4 + regresión logística
ojos.eval()
with torch.no_grad():
    car_pocos = ojos(X_pocos).numpy()       # características: 1568 números por foto
    car_np = ojos(X_np).numpy()
C = LogisticRegression(max_iter=2000).fit(car_pocos, y_pocos.numpy())
exa_C = C.score(car_np, y_np.numpy())

for nombre, exa in [("A. píxeles", exa_A), ("B. red desde cero", exa_B), ("C. transferencia", exa_C)]:
    print(f"{nombre:20} {exa:.1%}")

Los ojos entrenados con 0–4 **nunca vieron** un 5, un 7 o un 9, pero aprendieron a detectar trazos, curvas y esquinas. Esos detectores también sirven para los dígitos nuevos, y por eso la transferencia gana con tan pocos ejemplos.

✏️ *¿Por qué crees que la diferencia no es enorme aquí? Piensa en qué tan parecidos son los dígitos 0–4 y 5–9, y en qué tan fácil es MNIST.*

La transferencia brilla de verdad con fotos reales, donde aprender a ver desde cero cuesta millones de imágenes. Vamos allá.

## 2. Una red preentrenada de verdad: ResNet-18

`torchvision.models` trae redes famosas ya entrenadas. Usaremos **ResNet-18** (2015): una red convolucional de 18 capas, entrenada con **ImageNet**, un conjunto de **1.28 millones de fotos** de **1000 categorías** (animales, objetos, vehículos, comida…). Entrenarla tomó días con GPU; nosotros descargamos sus pesos en segundos.

In [ ]:
pesos = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=pesos)     # descarga unos 45 MB la primera vez
resnet.eval()

categorias = pesos.meta["categories"]
print(f"Parámetros: {sum(p.numel() for p in resnet.parameters()):,}")
print("Categorías:", len(categorias))
print("Algunas:", categorias[:5], "...", categorias[281:286])

¡11.7 millones de parámetros, más de 500 veces la red de RN 5! La red espera fotos **preparadas como en su entrenamiento**: de 224 × 224 píxeles y con los colores **normalizados** (a cada canal se le resta su promedio en ImageNet y se divide por su desviación, como la estandarización de ML 4). Los pesos traen esa preparación lista:

In [ ]:
preparar = pesos.transforms()
print(preparar)

Probemos con dos fotos que vienen incluidas en scikit-learn:

In [ ]:
from sklearn.datasets import load_sample_images
from PIL import Image

fotos = load_sample_images().images        # dos fotos a color, como arreglos de numpy

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
for eje, foto in zip(ejes, fotos):
    x = preparar(Image.fromarray(foto)).unsqueeze(0)    # forma (1, 3, 224, 224)
    with torch.no_grad():
        probs = torch.softmax(resnet(x), dim=1)[0]
    mejores = probs.topk(5)
    texto = "\n".join(f"{categorias[i]}: {p:.0%}" for p, i in zip(mejores.values, mejores.indices))
    eje.imshow(foto)
    eje.set_title(texto, fontsize=9, loc="left")
    eje.axis("off")
plt.tight_layout()
plt.show()

La red da sus **5 categorías más probables** (el *top-5*). ImageNet solo tiene esas 1000 categorías, así que la red responde con la más parecida que conoce.

✏️ *¿Te parecen razonables las respuestas? Para la segunda foto, ¿qué categoría habrías esperado? ¿Está entre las 1000? (Busca en `categorias` con `[c for c in categorias if "flower" in c or "daisy" in c]`.)*

### ¿Qué aprendió a ver?

La primera capa de ResNet-18 tiene 64 filtros de 7 × 7 sobre los 3 canales de color. Como en RN 5, podemos dibujarlos:

In [ ]:
filtros = resnet.conv1.weight.detach()          # forma (64, 3, 7, 7)
fig, ejes = plt.subplots(4, 16, figsize=(14, 3.8))
for k, eje in enumerate(ejes.flat):
    f = filtros[k].permute(1, 2, 0)              # (7, 7, 3) para dibujar a color
    f = (f - f.min()) / (f.max() - f.min())      # llevar a [0, 1]
    eje.imshow(f)
    eje.axis("off")
plt.suptitle("Los 64 filtros de la primera capa de ResNet-18")
plt.show()

Bordes en todas las direcciones, rayas finas y manchas de color: los mismos detectores que diseñamos a mano en RN 5, pero aprendidos con un millón de fotos. Las capas siguientes los combinan en texturas, partes (ojos, ruedas, hojas) y objetos.

## 3. Una tarea nueva con pocas fotos: CIFAR-10

**CIFAR-10** tiene fotos de 32 × 32 píxeles a color de 10 clases: avión, automóvil, pájaro, gato, venado, perro, rana, caballo, barco y camión. El conjunto completo tiene 50 000 fotos de entrenamiento, pero vamos a fingir que solo tenemos **50 fotos por clase** (500 en total), como pasaría en un proyecto real con fotos tomadas por nosotros.

La primera vez se descargan unos 170 MB (menos de un minuto en Colab). Cargamos dos versiones: las fotos de 32 × 32 tal cual, y las fotos preparadas para ResNet.

In [ ]:
cifar_32 = datasets.CIFAR10("datos", train=True, download=True, transform=transforms.ToTensor())
cifar_32_prueba = datasets.CIFAR10("datos", train=False, download=True, transform=transforms.ToTensor())
cifar_224 = datasets.CIFAR10("datos", train=True, download=True, transform=preparar)
cifar_224_prueba = datasets.CIFAR10("datos", train=False, download=True, transform=preparar)

clases = ["avión", "automóvil", "pájaro", "gato", "venado", "perro", "rana", "caballo", "barco", "camión"]
y_cifar = np.array(cifar_32.targets)
y_cifar_prueba = np.array(cifar_32_prueba.targets)

rng = np.random.default_rng(0)
idx_entreno = np.concatenate([rng.choice(np.where(y_cifar == c)[0], 50, replace=False) for c in range(10)])
idx_prueba = np.concatenate([rng.choice(np.where(y_cifar_prueba == c)[0], 100, replace=False) for c in range(10)])
print("Fotos para entrenar:", len(idx_entreno), "   fotos de prueba:", len(idx_prueba))

In [ ]:
fig, ejes = plt.subplots(3, 10, figsize=(14, 4.6))
for c in range(10):
    for fila in range(3):
        foto, etiqueta = cifar_32[idx_entreno[50 * c + fila]]
        ejes[fila, c].imshow(foto.permute(1, 2, 0))     # (3, 32, 32) → (32, 32, 3)
        ejes[fila, c].axis("off")
    ejes[0, c].set_title(clases[c], fontsize=10)
plt.suptitle("Algunas de nuestras 500 fotos de entrenamiento (32 × 32 píxeles)")
plt.show()

✏️ *Con solo 32 × 32 píxeles, ¿podrías distinguir siempre un gato de un perro, o un venado de un caballo?*

### Intento 1: una red convolucional desde cero

La red de RN 5, ahora con 3 canales de entrada (rojo, verde, azul) y fotos de 32 × 32, entrenada 30 épocas con Adam:

In [ ]:
X_ent32, y_ent = next(iter(DataLoader(Subset(cifar_32, idx_entreno), batch_size=500)))
X_pru32, y_pru = next(iter(DataLoader(Subset(cifar_32_prueba, idx_prueba), batch_size=1000)))

torch.manual_seed(0)
desde_cero = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 16 × 16 × 16
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 32 × 8 × 8
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 10),
)
optimizador = optim.Adam(desde_cero.parameters(), lr=0.001)
cargador = DataLoader(torch.utils.data.TensorDataset(X_ent32, y_ent), batch_size=64, shuffle=True)

for epoca in range(30):
    desde_cero.train()
    for X, y in cargador:
        perdida = criterio(desde_cero(X), y)
        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

desde_cero.eval()
with torch.no_grad():
    exa_cero = (desde_cero(X_pru32).argmax(dim=1) == y_pru).float().mean().item()
print(f"Red desde cero: {exa_cero:.1%} en prueba (adivinar al azar daría 10 %)")

Mejor que adivinar, pero bastante malo: con 500 fotos la red no alcanza a aprender a ver.

## 4. Intento 2: ResNet-18 como extractor de características

Idea: quitamos la **última capa** de ResNet-18 (la que elige entre las 1000 categorías de ImageNet). Lo que queda convierte cada foto en **512 números**: un resumen de lo que la red «ve» en la foto (texturas, partes, formas). A esos números los llamamos **características**.

En PyTorch, reemplazamos la última capa, `resnet.fc`, por `nn.Identity()`, una capa que no hace nada:

In [ ]:
print("Última capa original:", resnet.fc)
resnet.fc = nn.Identity()

with torch.no_grad():
    x = preparar(Image.fromarray(fotos[0])).unsqueeze(0)
    print("Características de la primera foto:", tuple(resnet(x).shape))

Ahora pasamos nuestras 500 fotos de entrenamiento y 1000 de prueba por ResNet. Es la parte lenta: cada foto atraviesa 18 capas a 224 × 224 píxeles. (Unos 30 segundos con CPU; menos de 10 con la GPU de Colab.)

In [ ]:
def extraer(conjunto, indices):
    resnet.eval()
    caracteristicas, etiquetas = [], []
    with torch.no_grad():                         # no vamos a entrenar ResNet
        for X, y in DataLoader(Subset(conjunto, indices), batch_size=100):
            caracteristicas.append(resnet(X))
            etiquetas.append(y)
    return torch.cat(caracteristicas), torch.cat(etiquetas)

inicio = time.time()
car_ent, y_ent = extraer(cifar_224, idx_entreno)
car_pru, y_pru = extraer(cifar_224_prueba, idx_prueba)
print(f"Listo en {time.time() - inicio:.0f} s.  Entrenamiento: {tuple(car_ent.shape)}   prueba: {tuple(car_pru.shape)}")

Encima de esas características entrenamos una **cabeza nueva**: una sola capa `nn.Linear(512, 10)`. Es lo único que aprende; ResNet no cambia. Como las características ya están calculadas, entrenar es instantáneo:

In [ ]:
torch.manual_seed(0)
cabeza = nn.Linear(512, 10)
optimizador = optim.Adam(cabeza.parameters(), lr=0.001)

for paso in range(300):
    perdida = criterio(cabeza(car_ent), y_ent)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

with torch.no_grad():
    pred_pru = cabeza(car_pru).argmax(dim=1)
exa_transfer = (pred_pru == y_pru).float().mean().item()

print(f"Red desde cero:  {exa_cero:.1%}")
print(f"Transferencia:   {exa_transfer:.1%}")
print(f"Parámetros entrenados: desde cero {sum(p.numel() for p in desde_cero.parameters()):,}   cabeza {sum(p.numel() for p in cabeza.parameters()):,}")

¡Con las **mismas 500 fotos**, la transferencia acierta el **doble**! Y entrenamos solo 5130 parámetros. Todo el trabajo difícil (aprender a ver) ya lo había hecho ResNet con ImageNet.

✏️ *ImageNet tiene muchas razas de perros y gatos, y también aviones, barcos y camiones. ¿Crees que eso ayuda? ¿Funcionaría igual para radiografías o para fotos de satélite?*

In [ ]:
fig, ejes = plt.subplots(2, 8, figsize=(14, 5.2))
for eje, i in zip(ejes.flat, range(0, 1000, 63)):
    foto = X_pru32[i].permute(1, 2, 0)
    bien = pred_pru[i] == y_pru[i]
    eje.imshow(foto)
    eje.set_title(f"{clases[pred_pru[i]]}" + ("" if bien else f"\n(era {clases[y_pru[i]]})"),
                  color="green" if bien else "red", fontsize=9)
    eje.axis("off")
plt.suptitle("Respuestas de la red con transferencia (verde = bien, rojo = mal)")
plt.tight_layout()
plt.show()

## 5. ¿Qué significan las 512 características? Vecinos cercanos

Si las características capturan «lo que hay en la foto», dos fotos parecidas deberían tener características **cercanas**. Comprobémoslo con numpy y la distancia de k vecinos (ML 4): para algunas fotos de prueba buscamos las 5 fotos de entrenamiento más cercanas, primero comparando **píxeles** y luego comparando **características**.

In [ ]:
P_ent = X_ent32.reshape(500, -1).numpy()       # píxeles: 3 · 32 · 32 = 3072 números
P_pru = X_pru32.reshape(1000, -1).numpy()
C_ent = car_ent.numpy()                        # características: 512 números
C_pru = car_pru.numpy()

def vecinos(a, B, k=5):
    distancias = np.sqrt(((B - a) ** 2).sum(axis=1))
    return np.argsort(distancias)[:k]

fig, ejes = plt.subplots(4, 6, figsize=(10, 7.2))
for fila, (i, tipo) in enumerate([(316, "píxeles"), (316, "características"), (720, "píxeles"), (720, "características")]):
    ejes[fila, 0].imshow(X_pru32[i].permute(1, 2, 0))
    ejes[fila, 0].set_title(f"buscada ({tipo})", fontsize=9)
    cercanos = vecinos(P_pru[i], P_ent) if tipo == "píxeles" else vecinos(C_pru[i], C_ent)
    for col, j in enumerate(cercanos, start=1):
        ejes[fila, col].imshow(X_ent32[j].permute(1, 2, 0))
        ejes[fila, col].set_title(clases[y_ent[j]], fontsize=9)
for eje in ejes.flat:
    eje.axis("off")
plt.tight_layout()
plt.show()

Con píxeles, los «vecinos» se parecen en **colores** y en el fondo, no en lo que muestran. Con características, se parecen en el **contenido**. Midámoslo: clasifiquemos cada foto de prueba con la clase de su vecino más cercano (k = 1).

In [ ]:
def exactitud_1vecino(entreno, prueba):
    aciertos = 0
    for i in range(len(prueba)):
        j = vecinos(prueba[i], entreno, k=1)[0]
        aciertos += y_ent[j] == y_pru[i]
    return aciertos.item() / len(prueba)

print(f"1 vecino con píxeles:          {exactitud_1vecino(P_ent, P_pru):.1%}")
print(f"1 vecino con características:  {exactitud_1vecino(C_ent, C_pru):.1%}")

Las características convierten cada foto en un punto de un espacio de 512 dimensiones donde **la distancia significa parecido**. A esa representación se le llama **embedding**, y la volveremos a encontrar con palabras en la próxima lección.

## 6. La receta completa en PyTorch: congelar y cambiar la cabeza

En la sección 4 calculamos las características aparte y entrenamos la cabeza por separado. La forma habitual en PyTorch es hacerlo todo dentro de **una sola red**:

1. Cargar la red preentrenada.
2. **Congelar** sus parámetros con `requires_grad = False`, para que el optimizador no los cambie.
3. Reemplazar la última capa por una nueva con el número de clases que queremos (sus parámetros sí aprenden).

In [ ]:
red_final = models.resnet18(weights=pesos)
for p in red_final.parameters():
    p.requires_grad = False                   # congelar todo
red_final.fc = nn.Linear(512, 10)             # cabeza nueva: estos sí aprenden

entrenables = sum(p.numel() for p in red_final.parameters() if p.requires_grad)
total = sum(p.numel() for p in red_final.parameters())
print(f"Parámetros entrenables: {entrenables:,} de {total:,} ({entrenables / total:.2%})")

# El optimizador solo recibe los parámetros que aprenden:
optimizador = optim.Adam(red_final.fc.parameters(), lr=0.001)

Entrenar esta red con las fotos es exactamente lo mismo que hicimos en la sección 4, solo que más lento, porque cada paso vuelve a pasar las fotos por ResNet. Comprobémoslo copiando en `red_final.fc` los pesos de nuestra `cabeza` ya entrenada:

In [ ]:
red_final.fc.load_state_dict(cabeza.state_dict())
red_final.eval()
X_algunas, y_algunas = next(iter(DataLoader(Subset(cifar_224_prueba, idx_prueba[:200]), batch_size=200)))
with torch.no_grad():
    pred = red_final(X_algunas).argmax(dim=1)
print(f"Exactitud en las primeras 200 fotos de prueba (aviones y automóviles): {(pred == y_algunas).float().mean():.1%}")
print("¿Mismas respuestas que la cabeza de la sección 4?", torch.equal(pred, pred_pru[:200]))

### Ajuste fino (*fine-tuning*)

Un paso más: después de entrenar la cabeza, se puede **descongelar** también la última parte de la red (`red_final.layer4`) y entrenarla con un paso muy pequeño (por ejemplo `lr=0.0001`). Así la red adapta sus detectores más avanzados a nuestras fotos. Suele mejorar unos puntos, pero es lento sin GPU y con pocas fotos puede sobreajustar (RN 6). Lo dejamos para el reto.

| Situación | Qué hacer |
|---|---|
| Pocas fotos, parecidas a ImageNet | congelar todo y entrenar solo la cabeza |
| Más fotos, o muy distintas a ImageNet | además, ajuste fino de las últimas capas con un paso pequeño |
| Muchísimas fotos y mucha GPU | se puede entrenar desde cero, pero casi siempre se empieza desde una red preentrenada |

## 7. Errores comunes

**Error 1: olvidar `red.eval()`.** ResNet usa capas de **normalización por lotes** (`BatchNorm`), que se comportan distinto en modo entrenamiento: usan el promedio del lote actual en vez del promedio de ImageNet. Con una sola foto en modo entrenamiento, ¡el resultado es otro!

In [ ]:
prueba_resnet = models.resnet18(weights=pesos)
x = preparar(Image.fromarray(fotos[0])).unsqueeze(0)
with torch.no_grad():
    prueba_resnet.eval()
    print("eval():  ", categorias[prueba_resnet(torch.cat([x, x])).argmax(dim=1)[0]])
    prueba_resnet.train()
    print("train(): ", categorias[prueba_resnet(torch.cat([x, x])).argmax(dim=1)[0]])

**Error 2: no preparar las fotos como en el entrenamiento original.** Sin el cambio de tamaño y la normalización de `pesos.transforms()`, la red recibe números en una escala que nunca vio. Pruébalo en *¿Qué pasa si…?*.

**Error 3: olvidar congelar.** Si no pones `requires_grad = False`, el optimizador intenta mover los 11 millones de parámetros con 500 fotos: es lento y puede **destruir** lo que la red sabía.

**Error 4: no cambiar el tamaño de la cabeza.** La cabeza original tiene 1000 salidas (las categorías de ImageNet). Para 10 clases, la nueva debe ser `nn.Linear(512, 10)`.

**Error 5: un número de entradas equivocado en la cabeza.** Cada red preentrenada tiene su propio tamaño de características: 512 en ResNet-18, 2048 en ResNet-50. Imprime `red.fc` antes de reemplazarla.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en la sección 1 usas 5 ejemplos por dígito en lugar de 10? ¿Y 50? ¿Se mantiene la ventaja de la transferencia?
2. …le pides a ResNet completa (con su cabeza de 1000 categorías) que clasifique una foto de gato de CIFAR-10? (Carga otra vez `models.resnet18(weights=pesos)` y usa `cifar_224_prueba[idx_prueba[300]]`.)
3. …usas `models.resnet18(weights=None)` (la misma arquitectura, pero con pesos **al azar**) como extractor de características? Repite las secciones 4 y la cabeza.
4. …entrenas la cabeza de la sección 4 con solo 10 fotos por clase? Pista: `car_ent[np.arange(500) % 50 < 10]` toma las 10 primeras de cada clase.
5. …extraes las características **sin** `preparar`, con las fotos de 32 × 32 tal cual (`cifar_32`)?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Contar parámetros.** ResNet-18 tiene 11 689 512 parámetros, de los cuales la cabeza original `nn.Linear(512, 1000)` tiene ¿cuántos? ¿Y la cabeza nueva `nn.Linear(512, 10)`? Calcula a mano y comprueba con `numel()`.

**Ejercicio 2 (fácil). Lo que ve ImageNet.** Busca en `categorias` todas las que contienen la palabra `"cat"` y todas las que contienen `"dog"`. ¿Cuántas hay? ¿Por qué crees que eso ayuda a distinguir gatos de perros en CIFAR-10?

**Ejercicio 3 (medio). Otra cabeza.** En lugar de `nn.Linear`, entrena una `LogisticRegression` de scikit-learn (ML 8) sobre `car_ent` y evalúala sobre `car_pru` (usa `.numpy()`). ¿Da una exactitud parecida? ¿Por qué tiene sentido que así sea?

**Ejercicio 4 (medio). ¿Cuántas fotos hacen falta?** Entrena la cabeza de la sección 4 con 2, 5, 10, 25 y 50 fotos por clase (usa la pista de *¿Qué pasa si…?* 4) y grafica la exactitud en prueba contra el número de fotos por clase. ¿Con cuántas fotos la transferencia ya supera a la red desde cero entrenada con 50?

**Ejercicio 5 (avanzado). ¿Dónde se confunde?** Haz la matriz de confusión (ML 7) de la red con transferencia en las 1000 fotos de prueba, con los nombres de `clases`. ¿Qué pares de clases confunde más? Muestra 6 fotos de la confusión más frecuente. ¿Te habrías equivocado tú?

## 🏆 Reto opcional: tu propio clasificador

1. **Ajuste fino.** Activa la GPU de Colab (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*). Toma `red_final` con la cabeza entrenada, descongela `red_final.layer4` (`for p in red_final.layer4.parameters(): p.requires_grad = True`) y entrénala unas 5 épocas con las 500 fotos, con `optim.Adam(..., lr=0.0001)` y aumento de datos (RN 6). Recuerda mover la red y cada lote a la GPU con `.to("cuda")`. ¿Cuánto mejora la exactitud?
2. **Tus fotos.** Toma con tu celular 20 fotos de cada uno de 3 objetos de tu casa o tu colegio (por ejemplo, tres frutas). Súbelas a Colab, prepáralas con `preparar(Image.open(ruta).convert("RGB"))`, extrae sus características y entrena una cabeza con 15 fotos por clase. Pruébala con las 5 restantes.

## 🎬 Cierre

Hoy aprendiste que:
- La **transferencia de aprendizaje** reutiliza una red entrenada en una tarea grande para resolver otra con **pocos datos**.
- Las primeras capas de una red aprenden detectores generales (bordes, colores, texturas) que sirven para muchas tareas.
- **ResNet-18**, entrenada con 1.28 millones de fotos de ImageNet, se carga con `models.resnet18(weights=...)` y espera fotos preparadas con `pesos.transforms()`.
- Quitando la última capa, la red convierte cada foto en **512 características**; con 500 fotos y una cabeza de 5130 parámetros pasamos de cerca de un tercio de aciertos a cerca de tres cuartos.
- En PyTorch se **congela** con `requires_grad = False` y se cambia la cabeza; el **ajuste fino** adapta también las últimas capas con un paso pequeño.
- En el espacio de características, **la distancia significa parecido**: es un **embedding** de la foto.

**Para pensar:** si una red puede convertir una foto en 512 números donde las fotos parecidas quedan cerca, ¿podríamos hacer lo mismo con **palabras**, de modo que «perro» quede cerca de «gato» y lejos de «avión»? En la próxima lección veremos **texto y embeddings**: cómo las redes convierten palabras en vectores y descubren solas relaciones entre ellas. 📝